# DS/CMPSC 410
# Spring 2026
# Lab 9 Fine Tuning T5 (a LLM) for Text Summarization
## Instructor: Prof. John Yen
## TA: Jingxi Zhu and Saptarshi Roy
# Lerarning Objects:
The learning objectives of this lab is for you to be able to
- Access a pretrained Encoder-Decoder LLM (T5-base) from Hugging Face.
- Preprocess text summarization data using truncation and padding.
- Fine-tune an Encoder-Decoder LLM (T5) for text summarization.
- Evaluate the performance of text summarization AFTER fine-tuning using ROUGE scores.
- Save the Text Summarization model and the tokenizer learned.
- Evaluate the Text Summarization model using testing data.
- Evaluate the Text Summarization model using a recent news article.
# Problems:
- Problem 1A: 10 points
- Problem 1B: 10 ponts
- Problem 2: 10 points
- Problem 3: 10 points
- Problem 4: 10 points
- Problem 5: 10 points
- Problem 6: 20 points
- Problem 7: 5 points
- Problem 8: 10 points
- Problem 9: 10 points
- Problem 10: 10 points
- Problem 11: 15 points
# Total: 130 Points
# Due: 11:59 pm, April 27 (Monday)

# Obtain a Hugging Face Access token (see slides for details)
# Copy the Access Token in a word file on your computer/laptop.
# Paste the Hugging Face Access Token to the "Secrets" page of your Google Colab.

# Make sure your Notebook is set up to use GPU in Google Colab for this lab:
- Under ``Edit``, click on ``Notbook Setting``
- Select a GPU resource (e.g., T4 GPU).

In [ ]:
!pip install rouge_score
!pip install evaluate
import numpy as np
import pandas as pd
import kagglehub
import re
from datasets import Dataset
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Load the Training, Testing, and Evaluation Dataset of CNN Daily Mail News Text Summarization (from Kaggle) to a directory (e.g., DS410Lab9) on your Google Drive (linked to your PSU account).
https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/My Drive/DS410/"

In [ ]:
path1="/content/drive/My Drive/DS410/train.csv"
training_DF = pd.read_table(path1, sep=",")
print(training_DF)

In [ ]:
path2="/content/drive/My Drive/DS410/validation.csv"
validation_DF = pd.read_table(path2, sep=",")
print(validation_DF)

In [ ]:
path3="/content/drive/My Drive/DS410/test.csv"
test_DF = pd.read_table(path3, sep=",")
print(test_DF)

# Drop the IDs so that duplicate articles from different sources (hence different ID) can be removed/dropped.
# Notice: ``drop`` is a mutable operation on Pandas Dataframe.  It modifies the input DataFrame.

In [ ]:
training_DF.drop(columns=['id'], inplace=True)
validation_DF.drop(columns=['id'], inplace=True)
test_DF.drop(columns=['id'], inplace=True)

# Now that the ID column has been removed, we can remove duplicate news entries (from different sources)

In [ ]:
training2_DF=training_DF.drop_duplicates()

In [ ]:
validation2_DF=validation_DF.drop_duplicates()
test2_DF = test_DF.drop_duplicates()

# Compare the size of the training, evaluation, and testing data before and after removing duplicates

In [ ]:
training_DF.shape

In [ ]:
print("Shape of Trainig, Validation, and Testing DataFrames before dopping duplicates:", training_DF.shape, validation_DF.shape, test_DF.shape)


In [ ]:
print("Shape of Trainig, Validation, and Testing DataFrames AFTER dopping duplicates:", training2_DF.shape, validation2_DF.shape, test2_DF.shape)


# Problem 1A (10 points)
## What is the impact of removing duplicates on the size of training data, validation data, and testing data?
- (i) Complete the code cell below (5 points)
- (ii) Answer the question above in the text cell below using percentage. (5 points)

In [ ]:
print("Shape of Trainig, Validation, and Testing DataFrames AFTER dopping duplicates:", training2_DF.shape, validation2_DF.shape, test2_DF.shape)

# Answer to Problem 1A (ii)
- Removing duplicates slightly reduced the size of the training and testing datasets, while the validation dataset remained unchanged.

- Training data decreased from 287,113 rows to 284,015 rows, which is a reduction of 3,098 rows (about 1.08%).
- Validation data remained the same at 13,368 rows, so the reduction is 0%.
- Testing data decreased from 11,490 rows to 11,488 rows, which is a reduction of 2 rows (about 0.02%).

This shows that most duplicate records were present in the training dataset, while the validation dataset contained no duplicates and the testing dataset had very few.

# Preprocessing: Replacing multiple space chacaters with a single space character.

# Problem 1B (10 points)
Complete the following code for cleaning extra space characters of training data, validation data, and testing data.

In [ ]:
def clean_spaces_text(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# An example of the input data (a news article).

In [ ]:
training2_DF.loc[0, 'article']

In [ ]:
training2_DF.columns

# Preprocess the training input data to remove extra spaces.


In [ ]:
training2_DF.loc[:,'article'] = training2_DF.loc[:, 'article'].apply(clean_spaces_text)

In [ ]:
validation2_DF.loc[:,'article'] = validation2_DF.loc[:, 'article'].apply(clean_spaces_text)

In [ ]:
test2_DF.loc[:,'article'] = test2_DF.loc[:, 'article'].apply(clean_spaces_text)

# Sample 10,000 training samples as the training set, 2,000 as validation data, and 2,000 as testing data set.

In [ ]:
t_size = 10000

sampled_training_DF = training2_DF.sample(n=t_size, random_state=7)



In [ ]:
sampled_validation_DF = validation2_DF.sample(n=2000, random_state=7)
sampled_test_DF = test2_DF.sample(n=2000, random_state=7)

# Convert the data sets into Hugging Face Data Format

In [ ]:
train_dataset = Dataset.from_pandas(sampled_training_DF)
test_dataset = Dataset.from_pandas(sampled_test_DF)

In [ ]:
validation_dataset = Dataset.from_pandas(sampled_validation_DF)

# If you have not done it yet, creare an Access Token for authentication with Hugging Face Hub
# After you obtain the token, save it in ``Secrets`` (key icon on the left panel of your ``Colab`` window) as the value of 'HF_TOKEN'.

# Problem 2 (10 points)
Access your Hugging Face Access Token ('HF_TOKEN') that you have saved in Google Colab.

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

# The Pre-trained Transformer Model we will use for this lab is ``T5-Base``.
## The model uses 220 million parameters.
## The model is pre-trained on the Colossal Clean Crawled Corpus (C4).
- Datasets used for unsupervoised denoising objective: C4 and Wiki-DPR
- Additional datasets are used for supervised text-to-text language modeling objective
[https://huggingface.co/google-t5/t5-base]

# Problem 3 (10 points)
Obtain the tokenizer from the pre-trained model `t5-base`.

In [ ]:
model_name = 't5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name,use_fast = True)

# Proprocess the Data for Text Summarization training and evlauation
- Input is tokenized articles with size ``max_input_length``
- Output is tokenized summary of the article with size ``max_target_length``

In [ ]:
max_input_length = 512
max_target_length = 128

def preprocess_data(data):
    model_inputs = tokenizer(
        data["article"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        data["highlights"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the ``proprocess_data`` to tokenize the training data, validation data, and testing data.

# Problem 4 (10 points)
Complete and execute the following code to map the function ``preprocess_data`` to ``train_dataset``, ``validation_dataset``, and ``test_dataset``.

In [ ]:
train_dataset2 = train_dataset.map(preprocess_data, batched=True)

In [ ]:
validation_dataset2 = validation_dataset.map(preprocess_data, batched=True)

In [ ]:
test_dataset2 = test_dataset.map(preprocess_data, batched=True)

# Use the pretrained T5-base (Transformer-based) Model for fine-tuning

# Problem 5 (10 points)
Load the pretrained T5-base model for sequence-to-sequence learning.
- Use a DataCollatorForSeq2Seq to handle dynamic padding and batching during training.¶
- Suppress unnecessary log messages for cleaner output.

In [ ]:
transformers.logging.set_verbosity_error()
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Defining Function for Evaluating Text Summarization Using ROUGE
## ROUGE score: Compare generated summary with summary provided by the original author, which are standard evaluation metrics for text summarization tasks.
- ``decoded_preds`` are decoded summaries generated by the model.
- ``decoded_labels`` are ``input-ids`` column in the original Pandas dataframe.
- The function ``rouge.compute`` uses the IDs (``decoded_labels``) to retrieve the actual ``summary`` for the input ``articles``, and compute ROUGE metrics (ROUGE-1, ROUGE-2, ROUGE-L) with stemming enabled.
- Return the F-measure for each metric.


In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    result = {k: v.mid.fmeasure if hasattr(v, "mid") else v for k, v in result.items()}
    return result

# Fine-tune the T5 LLM Model for Text Summarization

# Problem 6 (20 points)
Make sure you are in an area that has stable internet connection and you do not need to move before you run this code cell, because it takes a significant amount of time.  
Use 1e-4 (i.e., 1 * 10^(-4)) as the learning rate.

In [ ]:
model.gradient_checkpointing_enable()

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    # learning_rate=2e-5,
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=True,
    logging_dir="./logs",
    logging_steps=50,
    warmup_ratio=0.1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    disable_tqdm=False
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset2,
    eval_dataset=validation_dataset2,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Problem 7 (5 points)
Save the trained model, which we will use in a later problem.

In [ ]:
path_model="/content/drive/My Drive/DS410/modelT5_text_sum"
model.save_pretrained(path_model)
tokenizer.save_pretrained(path_model)

In [ ]:
#

# Problem 8 (10 points)
Evaluate the text summarization model using the testing data.

In [ ]:


print("Evaluation started...")
metrics = trainer.evaluate(eval_dataset=test_dataset2)
print("Evaluation finished.")
print(f"Test Metrics = {metrics}")



# Part B Using a Fine-tuned LLM model to create a summary for an input text.

In [ ]:
test2_DF.loc[1, 'article']

In [ ]:
test2_DF.loc[1, 'highlights']

# Problem 9 (10 points)
Reload the Text Summarization model and the Tokenizer you saved earlier. Use them to generate a summary for (a) the first test article, and (b) a recent news article.

In [ ]:
model_path = "/content/drive/My Drive/DS410/modelT5_text_sum"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# The input data of T5 needs to be preceeded by "summarize: " for text summarization task.

In [ ]:
news_article = test_DF.loc[1, 'article']
input_text = "summarize: " + news_article


In [ ]:
input_ids = tokenizer(input_text, return_tensors='pt').input_ids

In [ ]:
summary_ids = model.generate(
    input_ids,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

# Decode the output (summary) generated by the model
## Because the input has only one news article, its output is index 0 in summary_ids (outputs of T5).

In [ ]:
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
print(summary)

In [ ]:
news_article2 = "This week, OpenAI, the world’s largest AI startup, announced GPT 5.4-Cyber, a fine-tuned version of GPT-5.4 optimized for cyber capabilities. The company also said it would be scaling up its Trusted Access for Cyber (TAC) program to thousands of verified defenders and teams responsible for defending critical software. According to OpenAI’s announcement blog post, the TAC will be expanded to introduce access to GPT-5.4-Cyber for users willing to authenticate themselves as cybersecurity defenders. The company plans to use identity verification and KYC controls to govern who can access the new model. 'This is a version of GPT-5.4 which lowers the refusal boundary for legitimate cybersecurity work and enables new capabilities for advanced defensive workflows, including binary reverse engineering capabilities that enable security professionals to analyse compiled software for malware potential, vulnerabilities and security robustness without needing access to its source code,' the announcement blog post said. The announcement comes just a week after Anthropic unveiled Claude Mythos Preview and Project Glasswing, an initiative that provided selected organizations with access to the startup’s powerful new model to identify potential vulnerabilities in critical infrastructure."

In [ ]:
input_text2 = "summarize: " + news_article2

In [ ]:
input_ids2 = tokenizer(input_text2, return_tensors='pt').input_ids

In [ ]:
summary_ids2 = model.generate(
    input_ids2,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

In [ ]:
summary2 = tokenizer.decode(summary_ids2[0], skip_special_tokens=True)

In [ ]:
print(summary2)

# Problem 10 (10 points)
Select a recent newsartcile.
- (a) Provide the link to the news article. (2 points)
- (b) Copy the text to the code cell below and use the model to generate its summary. (8 points)

Solution to Problem 10 (a): Link to your news article: https://www.bbc.com/news/articles/c0q9xq7knq2o

In [ ]:
# code for Problem 10
recent_article = """
Courtesy of last night's Truth Social post from US President Donald Trump,
the ceasefire between Iran, the US and Israel which was due to expire on
Wednesday does at least persist.

Instead of fighting, we have a "war of blockades" over the Strait of Hormuz,
with both sides using force to intercept and seize commercial vessels.

The mood out in one of the world's most important waterways is combustible.
It would be unwise to bet against events spiralling out of control.

In the meantime, Islamabad still waits for Iranian and American representatives
to arrive for peace talks.

Parts of the city remain sealed off, the signs are still up and the hotel where
talks were expected to take place is empty, ready for the hoped-for return of
high-level delegations.

But after several days of fevered anticipation, the atmosphere has changed.
Gone is the talk of press pools in Washington heading to the airport, or
speculation about military transport planes arriving at a nearby airbase.
"""

inputs = tokenizer(
    recent_article,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

output = model.generate(
    inputs["input_ids"],
    max_length=80,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("Generated Summary:")
print(summary)

# Problem 11 (15 points)
In the answer cell below, for each text summarization task, whether the summaries generated in Problem are satisfactory, from your perspective. For each news summary, describe the positive and the negative aspects of the summary. (5 points for each news summary)

## Answer to Problem 11:

- Summary for the first test article:
The summary generated by the model is generally satisfactory. It captures the main idea of the article and includes the key event and outcome described in the story. One positive aspect is that the summary focuses on the most important information and is easy to understand. However, one negative aspect is that the model sometimes keeps extra wording from the original article, making the summary slightly longer than necessary.

- Summary for the BBC news article:
The summary generated by the model is partially satisfactory. A positive aspect is that it correctly identifies an important point in the article, which is the "war of blockades" in the Strait of Hormuz and the tense atmosphere in the region. This shows that the model was able to capture a key concept from the article. However, the summary is incomplete because it misses other important elements, such as the ceasefire situation and Pakistan’s efforts to organize peace talks between the US and Iran. Because of this, the summary does not fully represent the entire article.